# ✂️ Template: Train / Validation / Test Split

**Purpose:** Stratified three-way split with class imbalance audit, SMOTE option, and data leakage checks.  
**Default split:** 60 / 20 / 20 (train / val / test).  

**How to use:**
1. Set `TARGET` to your target column name.
2. Set `DROP_COLS` to any columns that should not be features (IDs, raw target strings, EDA bin columns).
3. Run all cells. The leakage audit will catch common mistakes automatically.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing   import LabelEncoder
from imblearn.over_sampling  import SMOTE
from collections             import Counter

# ── CHANGE THESE ─────────────────────────────────────────────────────────────
df           = pd.read_csv('your_cleaned_encoded_data.csv')
TARGET       = 'target'      # binary target column (0/1)
RANDOM_STATE = 42
TRAIN_SIZE   = 0.60
VAL_SIZE     = 0.20          # test gets the remainder

# Columns to drop before modeling (IDs, raw string targets, EDA bins, etc.)
DROP_COLS = [
    'id',                    # CHANGE: add your non-feature columns here
]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
# ─────────────────────────────────────────────────────────────────────────────


## 1️⃣  Encode Remaining Categorical Columns

In [ ]:
# Label-encode any remaining object/category columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c not in DROP_COLS + [TARGET]]

print(f'Encoding {len(cat_cols)} categorical columns: {cat_cols}')

df_enc         = df.copy()
label_encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    label_encoders[col] = le
    print(f'  {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')


## 2️⃣  Define X and y

In [ ]:
X = df_enc.drop(columns=DROP_COLS + [TARGET])
y = df_enc[TARGET]

print(f'Feature matrix : {X.shape[0]:,} rows × {X.shape[1]} columns')
print(f'Target vector  : {y.shape[0]:,} rows')
print(f'\nClass balance:')
print(y.value_counts(normalize=True).map('{:.2%}'.format))


## 3️⃣  Stratified Three-Way Split

In [ ]:
# Step 1: carve out test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size    = 1 - TRAIN_SIZE - VAL_SIZE,
    stratify     = y,
    random_state = RANDOM_STATE
)

# Step 2: split remainder into train and val
val_frac = VAL_SIZE / (TRAIN_SIZE + VAL_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size    = val_frac,
    stratify     = y_temp,
    random_state = RANDOM_STATE
)

print('='*55)
print('SPLIT SUMMARY')
print('='*55)
for name, Xs, ys in [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]:
    pct = len(Xs) / len(X) * 100
    dr  = ys.mean() * 100
    print(f'  {name:>6}: {len(Xs):>6,} rows ({pct:.0f}%)  |  Positive rate: {dr:.1f}%')
print(f'  TOTAL: {len(X):>6,}')


## 4️⃣  Class Imbalance Audit & SMOTE (optional)

In [ ]:
# Audit
print('Class distribution across splits:')
print('-'*50)
for name, y_s in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    c = Counter(y_s)
    ratio = c[1] / c[0]
    print(f'  {name:>6}: Neg={c[0]:,}  Pos={c[1]:,}  ratio={ratio:.3f}')

minority_pct = y_train.mean()
if minority_pct < 0.10:
    print(f'\n⚠️  Minority class is {minority_pct:.1%} — SMOTE recommended.')
else:
    print(f'\n✅  Minority class is {minority_pct:.1%} — class_weight="balanced" is sufficient.')

# ── SMOTE (disabled by default) ───────────────────────────────────────────────
APPLY_SMOTE = False   # CHANGE to True to enable

if APPLY_SMOTE:
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train, y_train = smote.fit_resample(X_train, y_train)
    print(f'\nPost-SMOTE training: {Counter(y_train)}')
    print(f'Rows: {len(X_temp):,} → {len(X_train):,}')
else:
    print('\nSMOTE not applied. Using class_weight="balanced" in estimators.')


## 5️⃣  Data Leakage Audit

In [ ]:
leakage_issues = []

# Check 1: No index overlap between splits
train_idx, val_idx, test_idx = set(X_train.index), set(X_val.index), set(X_test.index)
for pair, a, b in [('train∩val', train_idx, val_idx),
                   ('train∩test', train_idx, test_idx),
                   ('val∩test', val_idx, test_idx)]:
    overlap = a & b
    if overlap:
        leakage_issues.append(f'Index overlap [{pair}]: {len(overlap)} rows')
    else:
        print(f'✅  No overlap [{pair}]')

# Check 2: Target not in features
for col in [TARGET]:
    if col in X_train.columns:
        leakage_issues.append(f'"{col}" found in feature matrix — target leakage!')
    else:
        print(f'✅  "{col}" correctly excluded from X')

# Check 3: Row counts
total = len(X_train) + len(X_val) + len(X_test)
if total != len(X):
    leakage_issues.append(f'Row count mismatch: {total} ≠ {len(X)}')
else:
    print(f'✅  Row counts sum correctly: {len(X_train):,} + {len(X_val):,} + {len(X_test):,} = {total:,}')

print()
if leakage_issues:
    print('❌  LEAKAGE ISSUES DETECTED:')
    for issue in leakage_issues:
        print(f'    - {issue}')
else:
    print('✅  All leakage checks passed. Safe to proceed to modeling.')
